# – Part 2: Data Preprocessing


## Step 0 – Install Required Libraries

In [1]:
! gdown --id 1v00MuC_DxJx2Jsh785qEcu1mi7GTdD31

'gdown' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
!pip install camel-tools pyarabic nltk

In [5]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
from camel_tools.utils.normalize import normalize_alef_ar, normalize_alef_maksura_ar
from camel_tools.tokenizers.word import simple_word_tokenize
from pyarabic.araby import strip_tashkeel
import pyarabic.araby as araby
nltk.download('stopwords')
print('All libraries installed successfully!')

All libraries installed successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mohammad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


---
## Step 1a – Load and Explore the Dataset

In [8]:
FILE_PATH = r'C:\Users\Mohammad\Downloads\final_dataset_text_label.csv'

df = pd.read_csv(FILE_PATH)
print('Shape:', df.shape)
df.head()

Shape: (1495, 2)


,text,label
0,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,human
1,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,human
2,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,human
3,يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...,human
4,الأمن السيبراني هو أحد أهم المجالات في العصر ا...,human


In [10]:
# Basic exploration
print('=== Column names ===')
print(df.columns.tolist())

print('\n=== Label distribution ===')
print(df['label'].value_counts())

print('\n=== Missing values ===')
print(df.isnull().sum())


=== Column names ===
['text', 'label']

=== Label distribution ===
label
ai                     678
human                  660
ai generated            40
ai-generated            20
humen                   20
collected               15
human-written           10
ai_generation           10
ai_generated            10
hand written            10
human_written            5
human(collected)         5
human(hand_written)      5
collect                  5
label                    1
al                       1
Name: count, dtype: int64

=== Missing values ===
text     0
label    0
dtype: int64


---
## Step 1b – Label Normalization


In [13]:
#  Label Standardization

LABEL_MAP = {
    'ai'              : 'ai_generated',
    'ai generated'    : 'ai_generated',
    'ai-generated'    : 'ai_generated',
    'ai_generation'   : 'ai_generated',
    'ai_generated'    : 'ai_generated',
    'al'              : 'ai_generated',

    'human'               : 'human_written',
    'humen'               : 'human_written',
    'human-written'       : 'human_written',
    'human_written'       : 'human_written',
    'hand written'        : 'human_written',
    'human written'       : 'human_written',
    'human(hand_written)' : 'human_written',
    'human(collected)'    : 'human_written',
    'collect'             : 'human_written',
    'collected'           : 'human_written',
}

df['label'] = df['label'].astype(str).str.strip().str.lower()

before = len(df)
df = df[df['label'] != 'label'].reset_index(drop=True)
print(f'Removed {before - len(df)} row(s) with label="label"')

df['label'] = df['label'].map(LABEL_MAP)

unmapped = df['label'].isna().sum()
if unmapped > 0:
    print(f'Warning: {unmapped} row(s) had unrecognised labels and were dropped.')
    df = df.dropna(subset=['label']).reset_index(drop=True)

print('\n=== Clean Label Distribution ===')
print(df['label'].value_counts())
print(f'\nTotal samples after label cleaning: {len(df)}')


Removed 1 row(s) with label="label"

=== Clean Label Distribution ===
label
ai_generated     759
human_written    735
Name: count, dtype: int64

Total samples after label cleaning: 1494


---
## Step 2 – Text Cleaning



In [16]:
ARABIC_STOPWORDS = set(stopwords.words('arabic'))

EXTRA_STOPWORDS = {'هذا', 'هذه', 'ذلك', 'تلك', 'هناك', 'حيث', 'كما', 'أيضا',
                   'بعض', 'كل', 'جميع', 'أي', 'لكن', 'إلا', 'لأن', 'بسبب'}
ARABIC_STOPWORDS.update(EXTRA_STOPWORDS)

def clean_arabic_text(text: str) -> str:
    if not isinstance(text, str):
        return ''

    # Remove diacritics (tashkeel)
    text = araby.strip_tashkeel(text)

    # Remove tatweel (kashida elongation)
    text = araby.strip_tatweel(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove dates (DD/MM/YYYY, YYYY-MM-DD, etc.)
    text = re.sub(r'\b(\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4})\b', ' ', text)

    # Remove Arabic-Indic digit dates
    text = re.sub(
        r'[\u0660-\u0669]{1,4}[-/.]([\u0660-\u0669]{1,2})[-/.]([\u0660-\u0669]{1,4})',
        ' ', text)

    # Remove punctuation and ALL non-Arabic characters
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)

    # Remove remaining standalone digits (Arabic-Indic / Western)
    text = re.sub(r'[\u0660-\u0669\u06F0-\u06F90-9]+', ' ', text)

    # Remove Arabic punctuation marks (،  ؛  ؟  ۔  ٪)
    text = re.sub(r'[\u060C\u061B\u061F\u06D4\u066A\u066B\u066C]', ' ', text)

    # Remove extra whitespace / newlines
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stop words (optional, we will try both ways)
    #tokens = text.split()
    #tokens = [tok for tok in tokens if tok not in ARABIC_STOPWORDS]
    #text = ' '.join(tokens)

    return text


sample_raw = df['text'].iloc[9]
sample_clean = clean_arabic_text(sample_raw)
print('\n[Before]', sample_raw[:200])
print('\n[After] ', sample_clean[:200])



[Before] ظهر الأمن السيراني مع نهاية الحرب الباردة، وظهور مصطلح حرب الإنترنت أو الحرب السيبرانية، التي جاءت مع بداية اعتماد الدول على أجهزة الكمبيوتر في مؤسساتها وتطوير وحدة المعالجة المركزية في هذه الأجهزة، ا

[After]  ظهر الأمن السيراني مع نهاية الحرب الباردة وظهور مصطلح حرب الإنترنت أو الحرب السيبرانية التي جاءت مع بداية اعتماد الدول على أجهزة الكمبيوتر في مؤسساتها وتطوير وحدة المعالجة المركزية في هذه الأجهزة التي


In [18]:
# Apply cleaning to the entire dataset (after we check on sample)
df['text_clean'] = df['text'].apply(clean_arabic_text)

before = len(df)
df = df[df['text_clean'].str.strip() != ''].reset_index(drop=True)
after = len(df)
print(f'Rows before: {before} | Rows after: {after} | Dropped: {before - after}')
df[['text', 'text_clean', 'label']].head(3)

Rows before: 1494 | Rows after: 1494 | Dropped: 0


,text,text_clean,label
0,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,human_written
1,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,human_written
2,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,human_written


---
## Step 3 – Normalization (Word Variant Handling)


In [21]:
def normalize_arabic(text: str) -> str:
    """Normalize Arabic word variants. Returns a clean string."""
    if not isinstance(text, str):
        return ''

    # Normalize Alef variants → plain Alef
    text = re.sub(r'[أإآٱ]', 'ا', text)

    # Normalize Teh Marbuta → Heh
    text = text.replace('ة', 'ه')

    return text

df['text_normalized'] = df['text_clean'].apply(normalize_arabic)

print('text_normalized dtype:', df['text_normalized'].dtype)
print('Sample (string):', df['text_normalized'].iloc[0][:150])


text_normalized dtype: object
Sample (string): مع زياده الاعتماد على الخدمات الرقميه اصبحت حمايه البيانات الشخصيه لكل منا من اهم التحديات التي تواجهنا حيث تشهد الهجمات الالكترونيه تطورا سريعا في ال


---
## Step 4 – Tokenization


In [24]:
df['tokens_normalized'] = df['text_normalized'].apply(
    lambda t: simple_word_tokenize(t) if isinstance(t, str) else []
)
df['token_count'] = df['tokens_normalized'].apply(len)

print('\nSample tokens (first row):')
print(df['tokens_normalized'].iloc[0][:20])



Sample tokens (first row):
['مع', 'زياده', 'الاعتماد', 'على', 'الخدمات', 'الرقميه', 'اصبحت', 'حمايه', 'البيانات', 'الشخصيه', 'لكل', 'منا', 'من', 'اهم', 'التحديات', 'التي', 'تواجهنا', 'حيث', 'تشهد', 'الهجمات']


---
## Step 5 – Final Dataset Summary & Save

In [27]:
print('=== Final Dataset Overview ===')
print('\nDataFrame columns:')
print(df.columns.tolist())
df[['text', 'text_clean', 'text_normalized', 'tokens_normalized', 'label']].head(5)

=== Final Dataset Overview ===

DataFrame columns:
['text', 'label', 'text_clean', 'text_normalized', 'tokens_normalized', 'token_count']


,text,text_clean,text_normalized,tokens_normalized,label
0,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,مع زياده الاعتماد على الخدمات الرقميه اصبحت حم...,"[مع, زياده, الاعتماد, على, الخدمات, الرقميه, ا...",human_written
1,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,معظم الهجمات التي تحدث وتتم بسرعه وسهوله هي بس...,"[معظم, الهجمات, التي, تحدث, وتتم, بسرعه, وسهول...",human_written
2,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,"[في, ظل, التطور, التكنولوجي, الكبير, في, هذا, ...",human_written
3,يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...,يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...,يرتبط الامن السيبراني بالذكاء الاصطناعي ارتباط...,"[يرتبط, الامن, السيبراني, بالذكاء, الاصطناعي, ...",human_written
4,الأمن السيبراني هو أحد أهم المجالات في العصر ا...,الأمن السيبراني هو أحد أهم المجالات في العصر ا...,الامن السيبراني هو احد اهم المجالات في العصر ا...,"[الامن, السيبراني, هو, احد, اهم, المجالات, في,...",human_written


In [29]:
TEXT_OUTPUT_FILE = 'text_preprocessed_dataset.csv'
df[['text_normalized', 'label']].to_csv(TEXT_OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Saved to {TEXT_OUTPUT_FILE}')

TOKENS_OUTPUT_FILE = 'tokens_preprocessed_dataset.csv'
df[['tokens_normalized', 'label']].to_csv(TOKENS_OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Saved to {TOKENS_OUTPUT_FILE}')


Saved to text_preprocessed_dataset.csv
Saved to tokens_preprocessed_dataset.csv


# - PART 3: Text Representation 


In [32]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

In [33]:
X = df['text_clean']
y = df['label']
print("Number of samples:", len(X))
print(df[['text_clean', 'label']].head())

Number of samples: 1494
                                          text_clean          label
0  مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...  human_written
1  معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...  human_written
2  في ظل التطور التكنولوجي الكبير في هذا العصر وز...  human_written
3  يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...  human_written
4  الأمن السيبراني هو أحد أهم المجالات في العصر ا...  human_written


A) bag of words 

In [37]:
bow_vectorizer = CountVectorizer()

X_bow = bow_vectorizer.fit_transform(X)

# Shape of feature matrix
print("BoW Shape:", X_bow.shape)

print("\nSample Vocabulary:")
print(bow_vectorizer.get_feature_names_out()[:50])

# First document vector
print("\nFirst document vector:")
print(X_bow[0].toarray())



BoW Shape: (1494, 23614)

Sample Vocabulary:
['آب' 'آباء' 'آبائهم' 'آبي' 'آثار' 'آثارا' 'آثاره' 'آثارها' 'آجلا' 'آخذة'
 'آخر' 'آخرها' 'آخرون' 'آخرين' 'آر' 'آراء' 'آرائه' 'آرائهم' 'آرون' 'آسيا'
 'آفات' 'آفاق' 'آفاقا' 'آكل' 'آلات' 'آلاف' 'آلام' 'آلان' 'آلة' 'آله' 'آلي'
 'آليا' 'آليات' 'آلية' 'آليين' 'آمن' 'آمنا' 'آمنة' 'آن' 'آنثروبيك' 'آنفا'
 'آني' 'آنيا' 'آي' 'آية' 'آيفون' 'أباتشي' 'أبحاث' 'أبحاثنا' 'أبدأ']

First document vector:
[[0 0 0 ... 0 0 0]]


In [39]:
bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)
print(bow_df)

# Show first 5 rows and first 20 columns
#display(bow_df.iloc[:5, :20])


      آب  آباء  آبائهم  آبي  آثار  آثارا  آثاره  آثارها  آجلا  آخذة  ...  \
0      0     0       0    0     0      0      0       0     0     0  ...   
1      0     0       0    0     0      0      0       0     0     0  ...   
2      0     0       0    0     0      0      0       0     0     0  ...   
3      0     0       0    0     0      0      0       0     0     0  ...   
4      0     0       0    0     0      0      0       0     0     0  ...   
...   ..   ...     ...  ...   ...    ...    ...     ...   ...   ...  ...   
1489   0     0       0    0     0      0      0       0     0     0  ...   
1490   0     0       0    0     0      0      0       0     0     0  ...   
1491   0     0       0    0     0      0      0       0     0     0  ...   
1492   0     0       0    0     0      0      0       0     0     0  ...   
1493   0     0       0    0     0      0      0       0     0     0  ...   

      يومك  يومنا  يومه  يومهم  يومي  يوميا  يومياته  يومياتهم  يومية  ييا  
0        0

B) TF-IDF

In [42]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()

X_tfidf = tfidf_vectorizer.fit_transform(X)

print("TF-IDF Shape:", X_tfidf.shape)

print("\nSample Vocabulary:")
print(tfidf_vectorizer.get_feature_names_out()[:100])

# TF-IDF vector
print(X_tfidf.toarray())

TF-IDF Shape: (1494, 23614)

Sample Vocabulary:
['آب' 'آباء' 'آبائهم' 'آبي' 'آثار' 'آثارا' 'آثاره' 'آثارها' 'آجلا' 'آخذة'
 'آخر' 'آخرها' 'آخرون' 'آخرين' 'آر' 'آراء' 'آرائه' 'آرائهم' 'آرون' 'آسيا'
 'آفات' 'آفاق' 'آفاقا' 'آكل' 'آلات' 'آلاف' 'آلام' 'آلان' 'آلة' 'آله' 'آلي'
 'آليا' 'آليات' 'آلية' 'آليين' 'آمن' 'آمنا' 'آمنة' 'آن' 'آنثروبيك' 'آنفا'
 'آني' 'آنيا' 'آي' 'آية' 'آيفون' 'أباتشي' 'أبحاث' 'أبحاثنا' 'أبدأ' 'أبدا'
 'أبدت' 'أبدته' 'أبراج' 'أبرز' 'أبرزت' 'أبرزها' 'أبرمت' 'أبريل' 'أبسط'
 'أبطأ' 'أبطال' 'أبعاد' 'أبعادا' 'أبعد' 'أبعده' 'أبقى' 'أبل' 'أبلاه'
 'أبلش' 'أبلغ' 'أبناءك' 'أبنائك' 'أبنائهم' 'أبنه' 'أبهرت' 'أبوابا'
 'أبوابها' 'أبوظبي' 'أبون' 'أبونيت' 'أبية' 'أتابع' 'أتاح' 'أتاحت'
 'أتاناسوف' 'أتحدث' 'أتحمم' 'أتخيل' 'أتذكر' 'أترك' 'أتسائل' 'أتصفح'
 'أتطرق' 'أتطلع' 'أتعامل' 'أتعب' 'أتعلم' 'أتعود' 'أتفهم']
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [44]:
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

print("TF-IDF DataFrame:")
display(tfidf_df.iloc[:5, :20])


first_doc_tfidf = tfidf_df.iloc[0]
non_zero_tfidf = first_doc_tfidf[first_doc_tfidf > 0]
display(non_zero_tfidf.sort_values(ascending=False))

TF-IDF DataFrame:


,آب,آباء,آبائهم,آبي,آثار,آثارا,آثاره,آثارها,آجلا,آخذة,آخر,آخرها,آخرون,آخرين,آر,آراء,آرائه,آرائهم,آرون,آسيا
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


معلوماتنا    0.220807
او           0.202924
بهدف         0.167769
الشخصية      0.162757
يقوم         0.143635
               ...   
قد           0.035755
يمكن         0.034992
في           0.034883
مع           0.029457
هذه          0.024857
Name: 0, Length: 120, dtype: float64

C) WORD2VEC

In [47]:
# Tokenize text
tokenized_texts = [text.split() for text in X]

w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

# Vocabulary size
print("Vocabulary Size:")
print(len(w2v_model.wv))

Vocabulary Size:
11929


In [49]:
# Select one sample document
sample_doc = X.iloc[0]

print("ORIGINAL DOCUMENT")

print(sample_doc)

# Tokenize document
words = sample_doc.split()


print("WORD VECTORS BEFORE AVERAGING")
word_vectors_data = []

for word in words:
    if word in w2v_model.wv:
        vector = w2v_model.wv[word]

  # show first 10 dimensions(each word has 100 dim because vector size 100)
        row = [word] + list(vector[:10])

        word_vectors_data.append(row)

# Create DataFrame
columns = ['word'] + [f'dim_{i}' for i in range(10)]

word_vectors_df = pd.DataFrame(
    word_vectors_data,
    columns=columns
)

display(word_vectors_df)


print("DOCUMENT VECTOR AFTER AVERAGING")
vectors = [
    w2v_model.wv[word]
    for word in words
    if word in w2v_model.wv
]

# Average vectors
avg_vector = np.mean(vectors, axis=0)

# FUNCTION: Convert document → averaged vector

def document_vector(doc):

    words = doc.split()

    word_vectors = [
        w2v_model.wv[word]
        for word in words
        if word in w2v_model.wv
    ]

    # If document has no known words
    if len(word_vectors) == 0:
        return np.zeros(w2v_model.vector_size)

    # Average vectors
    return np.mean(word_vectors, axis=0)


# CREATE FULL WORD2VEC REPRESENTATION MATRIX
X_w2v = np.array([
    document_vector(text)
    for text in X
])

print("Word2Vec Representation Shape:")
print(X_w2v.shape)

# Convert to DataFrame for display
avg_vector_df = pd.DataFrame(
    avg_vector.reshape(1, -1)
)

# Show first 20 dimensions
display(avg_vector_df.iloc[:, :20])

ORIGINAL DOCUMENT
مع زيادة الاعتماد على الخدمات الرقمية أصبحت حماية البيانات الشخصية لكل منا من أهم التحديات التي تواجهنا حيث تشهد الهجمات الإلكترونية تطورا سريعا في السنوات الأخيرة حيث يستخدم المهاجمون تقنيات متقدمة لاختراق الأنظمة وسرقة البيانات وخاصة الحساسة منها الامن السيبراني هو الدرع الأول لحماية الخصوصية والمحافظة على المعلومات الخاصة والحساسة حيث يقوم على مجموعة من الأدوات والسياسات الأمنية وطرق إدارة المخاطر التي يمكن استخدامها في معظم الجهات سواء الحكومية او الخاصة وهو الطريقة التي نحمي بها بياناتها من الهجمات التي يقوم بها بعض المخربين بهدف الحصول على بيانات خاصة والوصول إليها بكل سهولة وبعد ذلك اما يتم استخدامها ضد الجهة التي تم اختراقها او حتى ابتزازها مقابل مبلغ من المال سواء كان ذلك بهدف التخريب او التجريب لذلك يجب علينا الحرص على معلوماتنا الشخصية وعدم اعطائها لأي شخص كان حتى لو كنا نثق به لانه بسبب اهمال بسيط قد تضيع كل هذه المعلومات وهناك عدة طرق للمحافظة على خصوصيتنا وحماية معلوماتنا الشخصية
WORD VECTORS BEFORE AVERAGING


,word,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9
0,مع,-0.014133,0.822817,0.726567,0.533145,-0.023732,-1.446628,0.471005,1.402077,-0.249189,-0.524399
1,زيادة,0.017386,0.652998,0.255868,0.332327,-0.083294,-0.930864,0.397629,0.991523,-0.302359,-0.286152
2,الاعتماد,-0.004109,0.508954,0.245264,0.376391,-0.223035,-0.662172,0.607043,0.809853,-0.525916,-0.264392
3,على,0.093382,0.679105,-0.131350,0.340555,0.074606,-0.873065,0.784609,1.179781,-0.775287,-0.297150
4,الخدمات,0.017087,0.631752,-0.103507,0.298776,-0.004011,-0.980199,0.583048,1.073857,-0.456057,-0.235584
...,...,...,...,...,...,...,...,...,...,...,...
143,على,0.093382,0.679105,-0.131350,0.340555,0.074606,-0.873065,0.784609,1.179781,-0.775287,-0.297150
144,خصوصيتنا,-0.008410,0.040684,0.009158,0.020526,-0.002825,-0.075384,0.036566,0.084641,-0.028403,-0.032550
145,وحماية,0.041474,0.473914,0.132661,0.214134,-0.023424,-0.705295,0.324787,0.776843,-0.241464,-0.226540
146,معلوماتنا,0.002615,0.059433,0.020267,0.039939,-0.003278,-0.099346,0.049361,0.111419,-0.027314,-0.036420


DOCUMENT VECTOR AFTER AVERAGING
Word2Vec Representation Shape:
(1494, 100)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.008532,0.496877,0.083714,0.251134,-0.030979,-0.761863,0.406236,0.827143,-0.346431,-0.239101,-0.160641,-0.622774,-0.207136,0.244637,0.30702,-0.162947,0.271856,-0.094973,-0.264792,-0.937142


In [51]:
# Save TF-IDF vectorizer
import pickle

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

print("TF-IDF vectorizer saved successfully!")

TF-IDF vectorizer saved successfully!


In [53]:
# Representation Summary
representation_summary = pd.DataFrame({

    'Representation': [
        'Bag of Words (BoW)',
        'TF-IDF',
        'Word2Vec'
    ],

    'Num_Documents': [
        X_bow.shape[0],
        X_tfidf.shape[0],
        X_w2v.shape[0]
    ],

    'Num_Features': [
        X_bow.shape[1],
        X_tfidf.shape[1],
        X_w2v.shape[1]
    ]

})

# Show table
display(representation_summary)

representation_summary.to_csv(
    'representation_summary.csv',
    index=False
)

print("representation_summary.csv saved successfully!")

,Representation,Num_Documents,Num_Features
0,Bag of Words (BoW),1494,23614
1,TF-IDF,1494,23614
2,Word2Vec,1494,100


representation_summary.csv saved successfully!


In [81]:
# ==================================================
# PART 4 : Classical Machine Learning Models
# ==================================================

from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd

In [83]:
# Results Table
results = []

In [85]:
# The dataset is split into 80% training, 10% validation, and 10% testing.
# We use stratified splitting to ensure that the class distribution
# (AI vs Human) remains balanced across the three subsets.

# ---------- BoW ----------

X_train_bow, X_temp_bow, y_train, y_temp = train_test_split(
    X_bow,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val_bow, X_test_bow, y_val, y_test = train_test_split(
    X_temp_bow,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# ---------- TF-IDF ----------

X_train_tfidf, X_temp_tfidf, _, _ = train_test_split(
    X_tfidf,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val_tfidf, X_test_tfidf, _, _ = train_test_split(
    X_temp_tfidf,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# ---------- Word2Vec ----------

X_train_w2v, X_temp_w2v, _, _ = train_test_split(
    X_w2v,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val_w2v, X_test_w2v, _, _ = train_test_split(
    X_temp_w2v,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train Samples :", len(y_train))
print("Validation Samples :", len(y_val))
print("Test Samples :", len(y_test))

  

Train Samples : 1195
Validation Samples : 149
Test Samples : 150


In [87]:
# ==================================================
# Evaluation Function
# ==================================================

def evaluate_model(model,
                   model_name,
                   representation_name,
                   X_train,
                   X_test,
                   y_train,
                   y_test):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average='weighted'
    )

    recall = recall_score(
        y_test,
        y_pred,
        average='weighted'
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average='weighted'
    )

    results.append([
        representation_name,
        model_name,
        accuracy,
        precision,
        recall,
        f1
    ])

    print(model_name)
    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)
       
         

In [89]:
evaluate_model(
    MultinomialNB(),
    "MultinomialNB",
    "BoW",
    X_train_bow,
    X_test_bow,
    y_train,
    y_test
)

MultinomialNB
Accuracy : 0.8066666666666666
Precision: 0.8379164190928896
Recall   : 0.8066666666666666
F1 Score : 0.8015847619047619


In [91]:
evaluate_model(
    SVC(kernel='rbf'),
    "SVM (RBF)",
    "BoW",
    X_train_bow,
    X_test_bow,
    y_train,
    y_test
)

SVM (RBF)
Accuracy : 0.7933333333333333
Precision: 0.7956578553018043
Recall   : 0.7933333333333333
F1 Score : 0.7927525759400509


In [93]:
evaluate_model(
    DecisionTreeClassifier(random_state=42),
    "Decision Tree",
    "BoW",
    X_train_bow,
    X_test_bow,
    y_train,
    y_test
)

Decision Tree
Accuracy : 0.68
Precision: 0.6807407407407408
Recall   : 0.68
F1 Score : 0.6798862019914651


In [95]:
evaluate_model(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "Random Forest",
    "BoW",
    X_train_bow,
    X_test_bow,
    y_train,
    y_test
)

Random Forest
Accuracy : 0.8133333333333334
Precision: 0.8278060969515243
Recall   : 0.8133333333333334
F1 Score : 0.8109090909090908


In [97]:
evaluate_model(
    AdaBoostClassifier(
        n_estimators=100,
        random_state=42,
        algorithm='SAMME'
    ),
    "AdaBoost",
    "BoW",
    X_train_bow,
    X_test_bow,
    y_train,
    y_test
)

AdaBoost
Accuracy : 0.76
Precision: 0.7602702702702703
Recall   : 0.76
F1 Score : 0.76


In [99]:
evaluate_model(
    MultinomialNB(),
    "MultinomialNB",
    "TF-IDF",
    X_train_tfidf,
    X_test_tfidf,
    y_train,
    y_test
)

MultinomialNB
Accuracy : 0.7533333333333333
Precision: 0.8132438278511226
Recall   : 0.7533333333333333
F1 Score : 0.7399420289855072


In [101]:
evaluate_model(
    SVC(kernel='rbf'),
    "SVM (RBF)",
    "TF-IDF",
    X_train_tfidf,
    X_test_tfidf,
    y_train,
    y_test
)

SVM (RBF)
Accuracy : 0.7866666666666666
Precision: 0.793763440860215
Recall   : 0.7866666666666666
F1 Score : 0.7850597800095648


In [103]:
evaluate_model(
    DecisionTreeClassifier(random_state=42),
    "Decision Tree",
    "TF-IDF",
    X_train_tfidf,
    X_test_tfidf,
    y_train,
    y_test
)

Decision Tree
Accuracy : 0.6333333333333333
Precision: 0.6334222222222222
Recall   : 0.6333333333333333
F1 Score : 0.6333496303539417


In [105]:
evaluate_model(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "Random Forest",
    "TF-IDF",
    X_train_tfidf,
    X_test_tfidf,
    y_train,
    y_test
)

Random Forest
Accuracy : 0.7866666666666666
Precision: 0.793763440860215
Recall   : 0.7866666666666666
F1 Score : 0.7850597800095648


In [107]:
evaluate_model(
    AdaBoostClassifier(
        n_estimators=100,
        random_state=42,
        algorithm='SAMME'
    ),
    "AdaBoost",
    "TF-IDF",
    X_train_tfidf,
    X_test_tfidf,
    y_train,
    y_test
)

AdaBoost
Accuracy : 0.6933333333333334
Precision: 0.6933333333333334
Recall   : 0.6933333333333334
F1 Score : 0.6933333333333334


In [109]:
 # Note:
# MultinomialNB is applied only on BoW and TF-IDF.
# It is not used with Word2Vec because Word2Vec produces continuous vectors
# that may contain negative values, while MultinomialNB is designed for
# non-negative count/frequency-based features.
#
# As a technical workaround, other approaches could be considered,
# such as transforming Word2Vec vectors into non-negative values
# (e.g., using MinMax scaling) or using a different Naive Bayes variant
# مثل GaussianNB. However, to stay consistent with the assignment
# requirement specifying MultinomialNB, we apply it only to BoW and TF-IDF.


evaluate_model(
    SVC(kernel='rbf'),
    "SVM (RBF)",
    "Word2Vec",
    X_train_w2v,
    X_test_w2v,
    y_train,
    y_test
)

SVM (RBF)
Accuracy : 0.66
Precision: 0.6600888888888888
Recall   : 0.66
F1 Score : 0.660015111782746


In [111]:
evaluate_model(
    DecisionTreeClassifier(random_state=42),
    "Decision Tree",
    "Word2Vec",
    X_train_w2v,
    X_test_w2v,
    y_train,
    y_test
)

Decision Tree
Accuracy : 0.6933333333333334
Precision: 0.6951226551226551
Recall   : 0.6933333333333334
F1 Score : 0.6922380952380952


In [113]:
evaluate_model(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "Random Forest",
    "Word2Vec",
    X_train_w2v,
    X_test_w2v,
    y_train,
    y_test
)

Random Forest
Accuracy : 0.7133333333333334
Precision: 0.7192877755264935
Recall   : 0.7133333333333334
F1 Score : 0.7108237934904601


In [115]:
evaluate_model(
    AdaBoostClassifier(
        n_estimators=100,
        random_state=42,
        algorithm='SAMME'
    ),
    "AdaBoost",
    "Word2Vec",
    X_train_w2v,
    X_test_w2v,
    y_train,
    y_test
)

AdaBoost
Accuracy : 0.7066666666666667
Precision: 0.7098885658914729
Recall   : 0.7066666666666667
F1 Score : 0.705092145285382


In [121]:
# ==================================================
# Final Comparison Table
# ==================================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Representation",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ]
)

# Sort by F1-score descending
results_df = results_df.sort_values(
    by="F1-score",
    ascending=False
).reset_index(drop=True)

display(results_df)

results_df.to_csv(
    "part4_results.csv",
    index=False
)

print("Results table saved successfully!")


,Representation,Model,Accuracy,Precision,Recall,F1-score
0,BoW,Random Forest,0.813333,0.827806,0.813333,0.810909
1,BoW,MultinomialNB,0.806667,0.837916,0.806667,0.801585
2,BoW,SVM (RBF),0.793333,0.795658,0.793333,0.792753
3,TF-IDF,SVM (RBF),0.786667,0.793763,0.786667,0.785060
4,TF-IDF,Random Forest,0.786667,0.793763,0.786667,0.785060
5,BoW,AdaBoost,0.760000,0.760270,0.760000,0.760000
6,TF-IDF,MultinomialNB,0.753333,0.813244,0.753333,0.739942
7,Word2Vec,Random Forest,0.713333,0.719288,0.713333,0.710824
8,Word2Vec,AdaBoost,0.706667,0.709889,0.706667,0.705092
9,TF-IDF,AdaBoost,0.693333,0.693333,0.693333,0.693333


Results table saved successfully!
